In [1]:
import pandas as pd
import numpy as np
import re, nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
df=pd.read_csv(r"C:\Users\dipay\Downloads\IMDB Dataset.csv\IMDB Dataset.csv")

In [3]:
print(df.shape)
print(df['sentiment'].value_counts())
print(df.info())
df.head()

(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
nltk.download('stopwords')
stop = set(stopwords.words('english'))
stem = PorterStemmer()
def preprocess(text):
    text = text.lower()                              
    text = re.sub(r'http\S+|www\S+', '', text)        
    text = re.sub(r'[^a-z\s]', '', text)              
    words = text.split()                             
    words = [w for w in words if w not in stop]       
    words = [stem.stem(w) for w in words]             
    return " ".join(words)
df['clean_text'] = df['review'].apply(preprocess)
df[['review','clean_text']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dipay\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod youll hook ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


In [5]:
bow = CountVectorizer()
X_bow = bow.fit_transform(df['clean_text'])
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df['clean_text'])
X_bow.shape, X_tfidf.shape

((50000, 137381), (50000, 137381))

In [6]:
tokens = [text.split() for text in df['clean_text']]
w2v = Word2Vec(tokens, vector_size=100, window=5, min_count=2)
w2v.wv['good']

array([ 1.2160183e+00, -1.5319822e+00,  4.8955029e-01, -1.1196315e+00,
        1.7988290e+00, -1.4072901e-01, -1.0900923e+00,  3.0174950e-01,
       -3.8299221e-01,  9.1644728e-01,  3.1752157e-01,  2.3543079e+00,
       -9.1558921e-01,  1.2570274e+00, -1.9565897e+00,  7.3146683e-01,
        1.5359490e-01,  1.9206583e+00,  2.5606298e-01,  1.0054320e+00,
       -1.0854034e+00, -9.9125767e-01,  2.7315933e-01,  1.0911120e+00,
       -6.8080026e-01, -8.4620732e-01, -2.0098634e+00,  9.7763389e-02,
       -5.3303945e-01, -9.0651202e-01,  1.6143301e+00,  1.6375200e+00,
       -1.4045429e+00,  3.4925258e-01, -1.5665771e-01, -6.4707571e-01,
        1.6747221e-01, -8.5067284e-01, -2.2254552e-01, -2.6406553e-01,
       -8.1265432e-01, -1.0613725e+00,  2.6891358e+00, -5.0123054e-01,
       -3.5533586e-01,  2.8393369e+00, -2.9608836e+00,  4.2859709e-01,
        5.0969130e-01,  3.6188093e-01,  9.9237996e-01,  3.7247694e-01,
       -1.0572542e+00, -4.1226250e-01,  3.4197223e-01, -1.2484518e+00,
      

In [7]:
def avg_w2v(words, model):
    vec = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vec, axis=0) if len(vec)>0 else np.zeros(100)
X_w2v = np.array([avg_w2v(text.split(), w2v) for text in df['clean_text']])
X_w2v.shape

(50000, 100)

In [8]:
from sklearn.model_selection import train_test_split
X = X_tfidf
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [9]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [10]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [11]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [12]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
models = {
    "Logistic Regression": lr,
    "Naive Bayes": nb,
    "Decision Tree": dt
}
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(name)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, pos_label='positive'))
    print("Recall:", recall_score(y_test, y_pred, pos_label='positive'))
    print("F1 Score:", f1_score(y_test, y_pred, pos_label='positive'))
    print("-" * 50)

Logistic Regression
Accuracy: 0.8919
Precision: 0.8809289509938988
Recall: 0.9038772213247173
F1 Score: 0.892255556663012
--------------------------------------------------
Naive Bayes
Accuracy: 0.8601
Precision: 0.8666666666666667
Recall: 0.847940226171244
F1 Score: 0.8572011840359294
--------------------------------------------------
Decision Tree
Accuracy: 0.7174
Precision: 0.711837385412515
Recall: 0.721324717285945
F1 Score: 0.7165496489468405
--------------------------------------------------


In [14]:
models = {"LR": lr, "NB": nb, "DT": dt}
results = []
for name, model in models.items():
    pred = model.predict(X_test)
    results.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred, average='weighted'),
        recall_score(y_test, pred, average='weighted'),
        f1_score(y_test, pred, average='weighted')
    ])
pd.DataFrame(results, columns=["Model","Accuracy","Precision","Recall","F1"])

,Model,Accuracy,Precision,Recall,F1
0,LR,0.8919,0.892188,0.8919,0.891895
1,NB,0.8601,0.860236,0.8601,0.860070
2,DT,0.7174,0.717474,0.7174,0.717406
